In [1]:
from src.data import *
plays, prof = load_raw()
df = build_interactions(plays, n_users=20000, seed=42)
df, country, users, arts = index_and_features(df, prof)
df = split_per_user(df, seed=42)

PROC.mkdir(parents=True, exist_ok=True)
df.to_parquet(PROC / 'interactions.parquet')
np.save(PROC / 'user_country.npy', country)

D = load_processed()
# verificações
tr, va, te = [set(zip(D[s]['u'], D[s]['a'])) for s in ('train', 'val', 'test')]
assert not (tr & va) and not (tr & te) and not (va & te)          # sem vazamento
assert set(D['val']['u']) == set(D['test']['u']) == set(range(D['nU']))
assert D['train'].groupby('u').size().min() >= 1

stats = pd.DataFrame({
    'usuários': [D['nU']], 'artistas': [D['nA']], 'interações': [len(df)],
    'esparsidade': [1 - len(df) / (D['nU'] * D['nA'])],
    'artistas/usuário (mediana)': [df.groupby('u').size().median()],
    'treino/val/teste': [' / '.join(str(len(D[s])) for s in ('train', 'val', 'test'))]})
stats.to_csv('results/estatisticas_dataset.csv', index=False); stats

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/usersha1-artmbid-artname-plays.tsv'

In [ ]:
from src.walks import *
indptr, indices = build_adj(D['train'], D['nU'], D['nA'])
C = random_walk_counts(indptr, indices, num_walks=50, walk_len=6, seed=0)

rows = []
for T in (10, 20):
    ei, ew = walk_graph(C, T)
    for d in (32, 64):
        for lr in (1e-3, 3e-3):
            m, h, secs = train_gnn(D, ei, ew, learn_weights=True, d=d, lr=lr)
            rows.append(dict(T=T, d=d, lr=lr, val_ndcg10=h['val_ndcg@10'].max(),
                             epocas=len(h), segundos=round(secs)))
pd.DataFrame(rows).to_csv('results/sensibilidade_hiperparametros.csv', index=False)